In [40]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score

import tensorflow 
from tensorflow import keras
from tensorflow.keras import layers
from keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.layers import GRU, Embedding

In [41]:
df = pd.read_csv(r"./dataset/archive/CEAS_08.csv")
df = df[["body", "label"]]
df.head()

,body,label
0,"Buck up, your troubles caused by small dimensi...",1
1,\nUpgrade your sex and pleasures with these te...,1
2,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1
3,Would anyone object to removing .so from this ...,0
4,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1


In [42]:
df.describe()

,label
count,39154.000000
mean,0.557848
std,0.496649
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


In [43]:
df["label"].value_counts()

label
1    21842
0    17312
Name: count, dtype: int64

In [44]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

def text_cleaning(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"httpwww?\w+", "", text)
    text = re.sub(r"https?\w+", "", text)
    text = re.sub(r"\s+", " ", text)
    
    final_text = []
    for word in text.split():
        if word in stop_words:
            final_text.append("")
        else:
            final_text.append(word)
    text = " ".join(final_text)

    
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["text"] = df["body"].apply(text_cleaning)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/synguyen446/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [45]:
tokens = Tokenizer()
tokens.fit_on_texts(df["text"])
vocab_size = len(tokens.word_index)
print("vocab_size: ", vocab_size)

vocab_size:  212350


In [46]:
seq = tokens.texts_to_sequences(df["text"])
padded_x = pad_sequences(seq, padding = "post", truncating = "post", maxlen = 200)
y = df["label"].values

In [47]:
x_train, x_test, y_train, y_test = train_test_split(padded_x, y, test_size = 0.2, random_state=42)


In [48]:

# Convert labels to float for binary crossentropy
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')
print(f"y_train dtype: {y_train.dtype}, shape: {y_train.shape}")
print(f"y_test dtype: {y_test.dtype}, shape: {y_test.shape}")
print(f"Unique values in y_train: {set(y_train)}")


y_train dtype: float32, shape: (31323,)
y_test dtype: float32, shape: (7831,)
Unique values in y_train: {np.float32(0.0), np.float32(1.0)}


In [54]:
model = Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=100, input_length=200),
    layers.GRU(
        units=128,
        activation="tanh",
        recurrent_activation="sigmoid",
        dropout=0.0,
        recurrent_dropout=0.0,
        return_sequences=False
    ),
    layers.Dense(1, activation="sigmoid")
])

In [55]:
model.compile(optimizer = "Adam", loss = "binary_crossentropy", metrics = ["accuracy", "precision"])

history = model.fit(
    x_train, y_train,
    epochs = 5,
    batch_size = 64,
    validation_data = (x_test, y_test)
)

Epoch 1/5


490/490 ━━━━━━━━━━━━━━━━━━━━ 15s 29ms/step - accuracy: 0.8401 - loss: 0.3076 - precision: 0.7960 - val_accuracy: 0.9932 - val_loss: 0.0260 - val_precision: 0.9926
Epoch 2/5
490/490 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.9959 - loss: 0.0161 - precision: 0.9954 - val_accuracy: 0.9958 - val_loss: 0.0167 - val_precision: 0.9961
Epoch 3/5
490/490 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.9990 - loss: 0.0049 - precision: 0.9985 - val_accuracy: 0.9953 - val_loss: 0.0186 - val_precision: 0.9961
Epoch 4/5
490/490 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.9984 - loss: 0.0065 - precision: 0.9985 - val_accuracy: 0.9949 - val_loss: 0.0180 - val_precision: 0.9958
Epoch 5/5
490/490 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.9996 - loss: 0.0026 - precision: 0.9993 - val_accuracy: 0.9954 - val_loss: 0.0219 - val_precision: 0.9956


In [56]:
model.save('model.keras')